# Telomere Analysis Pipeline

**Workflow:**
1. **Generate** - Build telomere_analysis CSV from CRAM files (FASTQ/FASTA accepted as fallback)
2. **Plot** - Run histograms, mutational signatures, Spearman correlations, pairwise heatmap, trendlines, and curve fitting

**Key parameters (Cell 5):**
- `REPEAT_K` — number of GGGTTA repeats (e.g. 3 → 18-base pattern). Single-base modifications are always counted in the **2nd repeat**.
- `CRAM_DIR` — directory containing one CRAM file per sample (no R1/R2 split).
- `REFERENCE_FASTA` — optional path to reference FASTA for CRAM decoding (set `None` for reference-free CRAMs).
- `OUTPUT_TXT` — plain-text summary written after the generate step.

In [33]:
# Install required packages (run once in UKB JupyterLab)
import subprocess, sys

packages = ['pysam', 'seaborn', 'scipy', 'matplotlib', 'pandas', 'numpy']
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', pkg])
print('All packages installed.')

All packages installed.


In [34]:
# Imports
import gzip
import json
from collections import defaultdict
import csv
import os
import sys
import numpy as np
import glob
import pysam
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from scipy.optimize import curve_fit
from scipy import stats
from scipy.stats import spearmanr, linregress
import numbers
from datetime import datetime

## Configuration

Set these paths to match your setup. Run from the **analysis** directory or adjust paths accordingly.

In [35]:
df = pd.read_csv("/mnt/project/Telomere/Telomere_length_age.csv")

print(df.head())

   Participant ID  Age at recruitment  Unadjusted T/S ratio | Instance 0  \
0         1000047                  66                           0.758285   
1         1000146                  64                           0.731674   
2         1000222                  69                           0.601980   
3         1000299                  59                           0.620696   
4         1000354                  65                                NaN   

   Unadjusted T/S ratio | Instance 1  Unadjusted T/S ratio | Instance 2  \
0                                NaN                                NaN   
1                                NaN                                NaN   
2                                NaN                                NaN   
3                                NaN                                NaN   
4                                NaN                                NaN   

   Adjusted T/S ratio | Instance 0  Adjusted T/S ratio | Instance 1  \
0                    

In [36]:
# UKB Configuration
# CRAM files should be downloaded to /opt/notebooks/
# pattern_generator.py should also be present in /opt/notebooks/ (or on sys.path)

import os

# Working directory on UKB JupyterLab
_ANALYSIS_DIR = "/opt/notebooks"

# Number of GGGTTA / CCCTAA repeat units used for pattern matching.
# Single-base modifications are always evaluated in the 2nd repeat.
# e.g. k=2 → GGGTTAGGGTTA, k=3 → GGGTTAGGGTTAGGGTTA
REPEAT_K = 3

# Directory containing CRAM files (one CRAM file per sample, no R1/R2 split).
# FASTQ/FASTA files are also accepted as a fallback if no CRAMs are found.
CRAM_DIR = _ANALYSIS_DIR

# Optional path to reference FASTA needed to decode CRAM files.
# Set to None if the CRAM was encoded without a reference (reference-free CRAM).
REFERENCE_FASTA = None  # e.g. "/mnt/project/reference/GRCh38.fa"

# No metadata file needed for UKB (age/length loaded from UKB CSV instead)
METADATA_PATH = None

# Output CSV path
CSV_OUT = os.path.join(_ANALYSIS_DIR, "telomere_analysis_ukb.csv")

# Output plain-text summary (per-sample read counts, telomere hits, mutation counts)
OUTPUT_TXT = os.path.join(_ANALYSIS_DIR, "output.txt")

# Output directories for plots
_TRENDLINES_DIR = os.path.join(_ANALYSIS_DIR, "trendlines")
_SPEARMAN_CORRELATIONS_DIR = os.path.join(_ANALYSIS_DIR, "spearman_correlations")
_HISTOGRAM_DIR = os.path.join(_ANALYSIS_DIR, "histograms")

print("Configuration complete")
print(f"Repeat k            : {REPEAT_K}  (patterns for {REPEAT_K}x GGGTTA repeats)")
print(f"Looking for CRAMs in: {CRAM_DIR}")
print(f"Reference FASTA     : {REFERENCE_FASTA or 'None (reference-free CRAM)'}")
print(f"Output CSV          : {CSV_OUT}")
print(f"Output summary TXT  : {OUTPUT_TXT}")


Configuration complete
Looking for FASTQ files in: /opt/notebooks
Output CSV: /opt/notebooks/telomere_analysis_ukb.csv


In [37]:
# Load UKB telomere length + age data to annotate samples after CSV generation
# This reads the Telomere_length_age.csv already in /mnt/project/Telomere/
import pandas as pd

ukb_df = pd.read_csv("/mnt/project/Telomere/Telomere_length_age.csv")
print("UKB metadata loaded:", ukb_df.shape)
print(ukb_df.head())


UKB metadata loaded: (502001, 14)
   Participant ID  Age at recruitment  Unadjusted T/S ratio | Instance 0  \
0         1000047                  66                           0.758285   
1         1000146                  64                           0.731674   
2         1000222                  69                           0.601980   
3         1000299                  59                           0.620696   
4         1000354                  65                                NaN   

   Unadjusted T/S ratio | Instance 1  Unadjusted T/S ratio | Instance 2  \
0                                NaN                                NaN   
1                                NaN                                NaN   
2                                NaN                                NaN   
3                                NaN                                NaN   
4                                NaN                                NaN   

   Adjusted T/S ratio | Instance 0  Adjusted T/S ratio | I

## Generate CSV

Programmatically build telomere patterns from `REPEAT_K`, scan CRAM files (one per sample),
count canonical and mutant telomere reads, and write the combined CSV + `output.txt` summary.

Single-base modifications are always the variants of the **2nd** GGGTTA/CCCTAA repeat within each k-repeat pattern.
No JSON patterns file is needed; no R1/R2 file pairing is required.

In [38]:
# ---------------------------------------------------------------------------
# Pattern generation — k-based, no JSON file required
# ---------------------------------------------------------------------------
# Try to import from the standalone module; fall back to inline definitions.
try:
    _pg_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    _pg_dir = _ANALYSIS_DIR
if _pg_dir not in sys.path:
    sys.path.insert(0, _pg_dir)

try:
    from pattern_generator import generate_patterns, G_REPEAT_UNIT, C_REPEAT_UNIT, GENERAL_MUTATION_MAP
    print("Loaded pattern_generator module from", _pg_dir)
except ImportError:
    # Inline fallback — identical logic to pattern_generator.py
    G_REPEAT_UNIT = "GGGTTA"
    C_REPEAT_UNIT = "CCCTAA"

    _G_STRAND_MUT_SPEC = {
        "G>A_g1": (0, "G", "A"), "G>A_g2": (1, "G", "A"), "G>A_g3": (2, "G", "A"),
        "G>C_g1": (0, "G", "C"), "G>C_g2": (1, "G", "C"), "G>C_g3": (2, "G", "C"),
        "G>T_g1": (0, "G", "T"), "G>T_g2": (1, "G", "T"), "G>T_g3": (2, "G", "T"),
        "T>A_t1": (3, "T", "A"), "T>A_t2": (4, "T", "A"),
        "T>C_t1": (3, "T", "C"), "T>C_t2": (4, "T", "C"),
        "T>G_t1": (3, "T", "G"), "T>G_t2": (4, "T", "G"),
        "A>T_a1": (5, "A", "T"), "A>G_a1": (5, "A", "G"), "A>C_a1": (5, "A", "C"),
    }
    _C_STRAND_MUT_SPEC = {
        "C>A_c1": (0, "C", "A"), "C>A_c2": (1, "C", "A"), "C>A_c3": (2, "C", "A"),
        "C>G_c1": (0, "C", "G"), "C>G_c2": (1, "C", "G"), "C>G_c3": (2, "C", "G"),
        "C>T_c1": (0, "C", "T"), "C>T_c2": (1, "C", "T"), "C>T_c3": (2, "C", "T"),
        "T>A_t1": (3, "T", "A"), "T>C_t1": (3, "T", "C"), "T>G_t1": (3, "T", "G"),
        "A>T_a1": (4, "A", "T"), "A>T_a2": (5, "A", "T"),
        "A>G_a1": (4, "A", "G"), "A>G_a2": (5, "A", "G"),
        "A>C_a1": (4, "A", "C"), "A>C_a2": (5, "A", "C"),
    }
    GENERAL_MUTATION_MAP = {
        "g_strand": {
            "G>A": ["G>A_g1", "G>A_g2", "G>A_g3"],
            "G>C": ["G>C_g1", "G>C_g2", "G>C_g3"],
            "G>T": ["G>T_g1", "G>T_g2", "G>T_g3"],
            "T>A": ["T>A_t1", "T>A_t2"], "T>C": ["T>C_t1", "T>C_t2"], "T>G": ["T>G_t1", "T>G_t2"],
            "A>T": ["A>T_a1"], "A>G": ["A>G_a1"], "A>C": ["A>C_a1"],
        },
        "c_strand": {
            "C>A": ["C>A_c1", "C>A_c2", "C>A_c3"],
            "C>G": ["C>G_c1", "C>G_c2", "C>G_c3"],
            "C>T": ["C>T_c1", "C>T_c2", "C>T_c3"],
            "T>A": ["T>A_t1"], "T>C": ["T>C_t1"], "T>G": ["T>G_t1"],
            "A>T": ["A>T_a1", "A>T_a2"], "A>G": ["A>G_a1", "A>G_a2"], "A>C": ["A>C_a1", "A>C_a2"],
        },
    }

    def generate_patterns(k: int):
        """Generate telomere patterns for k GGGTTA/CCCTAA repeats (inline fallback)."""
        if k < 2:
            raise ValueError(f"k must be >= 2; got {k}")
        g_can = G_REPEAT_UNIT * k
        c_can = C_REPEAT_UNIT * k
        off = len(G_REPEAT_UNIT)
        def _m(seq, pos, base):
            i = off + pos; return seq[:i] + base + seq[i+1:]
        patterns = {
            "g_strand": g_can, "c_strand": c_can,
            "g_strand_mutations": {sk: _m(g_can, p, b) for sk, (p, _, b) in _G_STRAND_MUT_SPEC.items()},
            "c_strand_mutations": {sk: _m(c_can, p, b) for sk, (p, _, b) in _C_STRAND_MUT_SPEC.items()},
        }
        return patterns, GENERAL_MUTATION_MAP, f"{k}x_repeat"

    print("pattern_generator module not found — using inline fallback definitions")


# ---------------------------------------------------------------------------
# CRAM file reading (pysam)
# ---------------------------------------------------------------------------

def read_cram_file(file_path: str, reference_fasta: str = None):
    """Read a CRAM (or BAM) file and yield query sequences using pysam."""
    kwargs = {}
    if reference_fasta:
        kwargs['reference_filename'] = reference_fasta
    with pysam.AlignmentFile(file_path, 'rc', **kwargs) as af:
        for read in af.fetch(until_eof=True):
            if read.query_sequence:
                yield read.query_sequence


def count_cram_reads(file_path: str, reference_fasta: str = None) -> int:
    """Count total reads in a CRAM file."""
    kwargs = {}
    if reference_fasta:
        kwargs['reference_filename'] = reference_fasta
    count = 0
    with pysam.AlignmentFile(file_path, 'rc', **kwargs) as af:
        for read in af.fetch(until_eof=True):
            if read.query_sequence:
                count += 1
    return count


def get_cram_files(directory: str):
    """Return sorted list of .cram files in the given directory."""
    return sorted(glob.glob(os.path.join(directory, "*.cram")))


# ---------------------------------------------------------------------------
# FASTQ / FASTA reading (kept as fallback)
# ---------------------------------------------------------------------------

def read_sequence_file(file_path: str):
    """Read FASTQ or FASTA file and yield sequences."""
    open_func = gzip.open if file_path.endswith('.gz') else open
    is_fasta = any(ext in file_path.lower() for ext in ['.fasta', '.fa', '.fas'])
    with open_func(file_path, 'rt') as f:
        if is_fasta:
            current_sequence = ""
            for line in f:
                line = line.strip()
                if line.startswith('>'):
                    if current_sequence:
                        yield current_sequence
                    current_sequence = ""
                else:
                    current_sequence += line
            if current_sequence:
                yield current_sequence
        else:
            while True:
                header = f.readline().strip()
                if not header:
                    break
                sequence = f.readline().strip()
                _ = f.readline()
                _ = f.readline()
                yield sequence


def get_sequence_files(directory: str):
    """Return sorted list of FASTQ/FASTA files (fallback when no CRAMs present)."""
    files = []
    for ext in ['*.fastq', '*.fastq.gz', '*.fasta', '*.fasta.gz', '*.fa', '*.fa.gz', '*.fas', '*.fas.gz']:
        files.extend(glob.glob(os.path.join(directory, ext)))
    return sorted(files)


def count_total_reads(file_path: str, reference_fasta: str = None) -> int:
    """Count total reads; handles CRAM, FASTQ, and FASTA."""
    if file_path.endswith('.cram'):
        return count_cram_reads(file_path, reference_fasta)
    open_func = gzip.open if file_path.endswith('.gz') else open
    count = 0
    with open_func(file_path, 'rt') as f:
        for _ in f:
            count += 1
    is_fasta = any(ext in file_path.lower() for ext in ['.fasta', '.fa', '.fas'])
    return count // 2 if is_fasta else count // 4


def count_patterns(sequence: str, pattern: str) -> int:
    return sequence.count(pattern)


# ---------------------------------------------------------------------------
# Metadata helpers (age / telomere length)
# ---------------------------------------------------------------------------

def load_age_data(metadata_file_path=None):
    """Load age data from metadata CSV (optional; returns {} if not found)."""
    age_data = {}
    if metadata_file_path is None:
        for candidate in [
            'greider_methods_table_s2_outliers_removed.csv',
            '../analysis/greider_methods_table_s2_outliers_removed.csv',
            'analysis/greider_methods_table_s2_outliers_removed.csv',
        ]:
            if os.path.exists(candidate):
                metadata_file_path = candidate
                break
        else:
            return age_data
    with open(metadata_file_path, 'r') as f:
        reader = csv.DictReader(f)
        for row in reader:
            name = row['fastq file name'].replace('_', '.')
            age_data[name] = row['Age (Years)']
    return age_data


def load_length_data(metadata_file_path=None):
    """Load telomere length data from metadata CSV (optional; returns {} if not found)."""
    length_data = {}
    if metadata_file_path is None:
        for candidate in [
            'greider_methods_table_s2_outliers_removed.csv',
            '../analysis/greider_methods_table_s2_outliers_removed.csv',
            'analysis/greider_methods_table_s2_outliers_removed.csv',
        ]:
            if os.path.exists(candidate):
                metadata_file_path = candidate
                break
        else:
            return length_data
    with open(metadata_file_path, 'r') as f:
        reader = csv.DictReader(f)
        for row in reader:
            name = row['fastq file name'].replace('_', '.')
            length_data[name] = row['Mean Telomere Length (bps)']
    return length_data

In [39]:
def _write_output_summary(txt_path, summary_rows, version, k, data_dir, csv_path,
                           file_type, patterns, general_mutation_map):
    """Write a human-readable plain-text summary of the analysis run."""
    total_reads = sum(r['total_reads'] for r in summary_rows)
    total_c = sum(r['c_strand'] for r in summary_rows)
    total_g = sum(r['g_strand'] for r in summary_rows)
    total_telomere = total_c + total_g
    total_mutations = sum(r['total_mutations'] for r in summary_rows)

    with open(txt_path, 'w') as f:
        f.write("=" * 72 + "\n")
        f.write("TELOMERE ANALYSIS — OUTPUT SUMMARY\n")
        f.write("=" * 72 + "\n")
        f.write(f"Run date/time       : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Pattern version     : {version}  (k={k} repeats of GGGTTA/CCCTAA)\n")
        f.write(f"Input directory     : {data_dir}\n")
        f.write(f"Input file type     : {file_type}\n")
        f.write(f"Output CSV          : {csv_path}\n")
        f.write(f"Output TXT          : {txt_path}\n")
        f.write("\n")
        f.write("CANONICAL PATTERNS\n")
        f.write(f"  G-strand ({k}x): {patterns['g_strand']}\n")
        f.write(f"  C-strand ({k}x): {patterns['c_strand']}\n")
        f.write(f"  Repeat unit : G-strand = {G_REPEAT_UNIT}  |  C-strand = {C_REPEAT_UNIT}\n")
        f.write(f"  Mutations evaluated in the 2nd repeat (positions 6–11)\n")
        f.write(f"  G-strand variants : {len(patterns['g_strand_mutations'])}\n")
        f.write(f"  C-strand variants : {len(patterns['c_strand_mutations'])}\n")
        f.write("\n")
        f.write("SINGLE-BASE MODIFICATION VARIANTS — G-strand (2nd repeat)\n")
        for subkey, seq in patterns['g_strand_mutations'].items():
            f.write(f"  {subkey:<12}: {seq}\n")
        f.write("\n")
        f.write("SINGLE-BASE MODIFICATION VARIANTS — C-strand (2nd repeat)\n")
        for subkey, seq in patterns['c_strand_mutations'].items():
            f.write(f"  {subkey:<12}: {seq}\n")
        f.write("\n")
        f.write("=" * 72 + "\n")
        f.write("RUN SUMMARY\n")
        f.write("=" * 72 + "\n")
        f.write(f"Samples processed       : {len(summary_rows)}\n")
        f.write(f"Total reads             : {total_reads:,}\n")
        f.write(f"Total telomere reads    : {total_telomere:,}  (c-strand + g-strand hits)\n")
        f.write(f"  C-strand hits         : {total_c:,}\n")
        f.write(f"  G-strand hits         : {total_g:,}\n")
        f.write(f"Total mutations found   : {total_mutations:,}\n")
        if total_reads > 0:
            f.write(f"Telomere read fraction  : {total_telomere / total_reads:.6f}\n")
        f.write("\n")
        f.write("PER-SAMPLE RESULTS\n")
        f.write("-" * 72 + "\n")
        f.write(f"{'Sample':<22} {'Total Reads':>13} {'C-strand':>10} {'G-strand':>10} {'Mutations':>10}\n")
        f.write("-" * 72 + "\n")
        for r in summary_rows:
            f.write(
                f"{r['sample_id']:<22} {r['total_reads']:>13,} "
                f"{r['c_strand']:>10} {r['g_strand']:>10} {r['total_mutations']:>10}\n"
            )
        f.write("-" * 72 + "\n")
        f.write(
            f"{'TOTAL':<22} {total_reads:>13,} {total_c:>10} {total_g:>10} {total_mutations:>10}\n"
        )
        f.write("\n")
        f.write("MUTATION TYPE GROUPINGS\n")
        f.write("-" * 72 + "\n")
        for strand, mutmap in general_mutation_map.items():
            f.write(f"\n{strand}:\n")
            for mut_type, subkeys in mutmap.items():
                f.write(f"  {mut_type}: {', '.join(subkeys)}\n")
        f.write("\n")
        f.write("=" * 72 + "\n")
        f.write("END OF SUMMARY\n")
        f.write("=" * 72 + "\n")


def generate_csv(
    data_dir: str,
    k: int = 3,
    reference_fasta: str = None,
    output_callback=None,
    metadata_file_path=None,
    output_csv_path=None,
    output_txt_path=None,
):
    """
    Generate a single CSV file from CRAM (or FASTQ/FASTA) telomere data.

    Each CRAM file is treated as one independent sample — no R1/R2 separation.
    Patterns are generated programmatically from the repeat count k.
    Writes an output.txt summary alongside the CSV when output_txt_path is given.
    """
    patterns, general_mutation_map, version = generate_patterns(k)

    if output_csv_path is None:
        safe_version = version.replace(' ', '_')
        csv_path = f"telomere_analysis_{safe_version}.csv"
    else:
        csv_path = output_csv_path
        parent = os.path.dirname(csv_path)
        if parent:
            os.makedirs(parent, exist_ok=True)

    # Prefer CRAM files; fall back to FASTQ/FASTA
    cram_files = get_cram_files(data_dir)
    fastq_files = get_sequence_files(data_dir)

    if cram_files:
        input_files = cram_files
        file_type = "CRAM"
    elif fastq_files:
        input_files = fastq_files
        file_type = "FASTQ/FASTA"
    else:
        msg = f"No CRAM or FASTQ/FASTA files found in {data_dir}"
        print(msg)
        if output_callback:
            output_callback(msg)
        return None

    msg = f"Found {len(input_files)} {file_type} file(s) in {data_dir}"
    print(msg)
    if output_callback:
        output_callback(msg)

    age_data = load_age_data(metadata_file_path)
    length_data = load_length_data(metadata_file_path)

    mutation_keys = []
    for group in ['g_strand_mutations', 'c_strand_mutations']:
        for subkey in patterns[group].keys():
            mutation_keys.append(f"{group}_{subkey}")

    fieldnames = ['FileName', 'Age', 'Telomere_Length', 'Total_Reads', 'c_strand', 'g_strand']
    fieldnames.extend(mutation_keys)
    fieldnames.extend([f"{mk}_per_1k" for mk in mutation_keys])
    general_mutation_headers = []
    for strand, mutmap in general_mutation_map.items():
        for mut in mutmap:
            general_mutation_headers.append(f"{strand}_{mut}_per_1k")
    fieldnames.extend(general_mutation_headers)
    fieldnames.extend([
        'composite_transition_per_1k', 'composite_transversion_per_1k',
        'g_strand_mutations_sum_per_1k', 'c_strand_mutations_sum_per_1k',
        'log_telomere_length', 'telomere_length_bin', 'mutation_rate_normalized_by_length',
    ])
    for strand, mutmap in general_mutation_map.items():
        for mut in mutmap:
            fieldnames.append(f"{strand}_{mut}_sum_per_1k")
    fieldnames.append('total_mutations_per_1k_strand_specific')
    fieldnames.extend([
        'total_mutations_over_total_g_strand_per_1k',
        'total_mutations_over_total_c_strand_per_1k',
    ])

    summary_rows = []

    with open(csv_path, 'w', newline='') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()

        for file_path in input_files:
            filename = os.path.basename(file_path)

            # Skip reference genome files
            if any(ref in filename for ref in ['GRCh38', 'hg38', 'reference']):
                continue

            # Derive sample ID by stripping the file extension (no R1/R2 removal needed)
            sample_id = filename
            for ext in ['.cram', '.fastq.gz', '.fastq', '.fasta.gz', '.fasta', '.fa.gz', '.fa', '.fas.gz', '.fas']:
                if sample_id.endswith(ext):
                    sample_id = sample_id[:-len(ext)]
                    break

            counts = defaultdict(int)
            total_reads = count_total_reads(file_path, reference_fasta)

            seq_iter = read_cram_file(file_path, reference_fasta) if file_path.endswith('.cram') else read_sequence_file(file_path)

            for sequence in seq_iter:
                counts['c_strand'] += count_patterns(sequence, patterns['c_strand'])
                counts['g_strand'] += count_patterns(sequence, patterns['g_strand'])
                for group in ['g_strand_mutations', 'c_strand_mutations']:
                    for subkey, subpattern in patterns[group].items():
                        counts[f"{group}_{subkey}"] += count_patterns(sequence, subpattern)

            age = age_data.get(sample_id, '')
            length = length_data.get(sample_id, '')
            g_strand_total = counts['g_strand']
            c_strand_total = counts['c_strand']
            g_strand_mutations_total = sum(counts[mk] for mk in counts if mk.startswith('g_strand_mutations_'))
            c_strand_mutations_total = sum(counts[mk] for mk in counts if mk.startswith('c_strand_mutations_'))
            g_strand_normalizer = g_strand_total + g_strand_mutations_total
            c_strand_normalizer = c_strand_total + c_strand_mutations_total

            def per_1k_ss(val, norm):
                return (val / norm) * 1000 if norm > 0 else 0

            row = {
                'FileName': sample_id, 'Age': age, 'Telomere_Length': length,
                'Total_Reads': total_reads,
                'c_strand': c_strand_total, 'g_strand': g_strand_total,
            }
            for mk in mutation_keys:
                row[mk] = counts.get(mk, 0)
                norm = g_strand_normalizer if mk.startswith('g_strand_mutations_') else c_strand_normalizer
                row[f"{mk}_per_1k"] = per_1k_ss(counts.get(mk, 0), norm)
            for strand, mutmap in general_mutation_map.items():
                norm = g_strand_normalizer if strand == 'g_strand' else c_strand_normalizer
                for mut, subtypes in mutmap.items():
                    total = sum(counts.get(f"{strand}_mutations_{st}", 0) for st in subtypes)
                    row[f"{strand}_{mut}_per_1k"] = per_1k_ss(total, norm)
            for strand, mutmap in general_mutation_map.items():
                for mut, subtypes in mutmap.items():
                    row[f"{strand}_{mut}_sum_per_1k"] = sum(
                        row.get(f"{strand}_mutations_{st}_per_1k", 0) for st in subtypes
                    )
            total_mutations = sum(counts[mk] for mk in mutation_keys)
            if total_mutations > 0 and (g_strand_normalizer > 0 or c_strand_normalizer > 0):
                g_w = g_strand_mutations_total / total_mutations
                c_w = c_strand_mutations_total / total_mutations
                w_norm = g_w * g_strand_normalizer + c_w * c_strand_normalizer
                row['total_mutations_per_1k_strand_specific'] = per_1k_ss(total_mutations, w_norm)
            else:
                row['total_mutations_per_1k_strand_specific'] = 0
            row['total_mutations_over_total_g_strand_per_1k'] = per_1k_ss(total_mutations, g_strand_normalizer)
            row['total_mutations_over_total_c_strand_per_1k'] = per_1k_ss(total_mutations, c_strand_normalizer)
            row['g_strand_mutations_sum_per_1k'] = sum(
                v for rk, v in row.items() if rk.startswith('g_strand_mutations') and rk.endswith('_per_1k')
            )
            row['c_strand_mutations_sum_per_1k'] = sum(
                v for rk, v in row.items() if rk.startswith('c_strand_mutations') and rk.endswith('_per_1k')
            )
            try:
                row['log_telomere_length'] = np.log1p(float(length)) if length else 0
            except Exception:
                row['log_telomere_length'] = 0
            try:
                lv = float(length)
                row['telomere_length_bin'] = 'short' if lv < 5000 else ('medium' if lv < 8000 else 'long')
            except Exception:
                row['telomere_length_bin'] = 'unknown'
            try:
                lv = float(length)
                row['mutation_rate_normalized_by_length'] = row['g_strand_mutations_sum_per_1k'] / lv if lv > 0 else 0
            except Exception:
                row['mutation_rate_normalized_by_length'] = 0

            writer.writerow(row)

            messages = [
                f"\nSample  : {sample_id}  ({file_type}: {filename})",
                f"  Age              : {age}",
                f"  Telomere Length  : {length}",
                f"  Total Reads      : {total_reads:,}",
                f"  {version} c-strand: {c_strand_total}",
                f"  {version} g-strand: {g_strand_total}",
                f"  G-strand mutations: {g_strand_mutations_total}",
                f"  C-strand mutations: {c_strand_mutations_total}",
                f"  G-strand normalizer: {g_strand_normalizer}",
                f"  C-strand normalizer: {c_strand_normalizer}",
                f"  Total mutations  : {total_mutations}",
            ]
            if g_strand_total == 0 and c_strand_total == 0:
                messages.append(f"  WARNING: No telomere sequences found in {sample_id}")
            for msg in messages:
                print(msg)
                if output_callback:
                    output_callback(msg)

            summary_rows.append({
                'sample_id': sample_id, 'file': filename,
                'total_reads': total_reads,
                'c_strand': c_strand_total, 'g_strand': g_strand_total,
                'g_strand_mutations': g_strand_mutations_total,
                'c_strand_mutations': c_strand_mutations_total,
                'total_mutations': total_mutations,
                'age': age, 'telomere_length': length,
            })

    # Write output.txt summary
    if output_txt_path and summary_rows:
        _write_output_summary(
            output_txt_path, summary_rows, version, k, data_dir, csv_path,
            file_type, patterns, general_mutation_map,
        )
        print(f"\nSummary written to: {output_txt_path}")

    return csv_path

In [ ]:
import time

print("[1/2] Generating telomere_analysis CSV from UKB CRAM files...")
print(f"  Repeat k            : {REPEAT_K}")
print(f"  Input directory     : {CRAM_DIR}")
print(f"  Reference FASTA     : {REFERENCE_FASTA or 'None (reference-free)'}")

start_time = time.time()

csv_path_result = generate_csv(
    data_dir=CRAM_DIR,
    k=REPEAT_K,
    reference_fasta=REFERENCE_FASTA,
    metadata_file_path=METADATA_PATH,
    output_csv_path=CSV_OUT,
    output_txt_path=OUTPUT_TXT,
)

elapsed = time.time() - start_time
minutes, seconds = divmod(elapsed, 60)
print(f"\nTotal runtime: {int(minutes)}m {seconds:.2f}s")

csv_path = csv_path_result if csv_path_result else CSV_OUT
print(f"CSV written to    : {csv_path}")
print(f"Summary written to: {OUTPUT_TXT}")

[1/2] Generating telomere_analysis CSV from UKB FASTQ files...
Scanning for FASTQ files in: /opt/notebooks

Processing sample: 1088576 (files: 1088576_R1.fastq.gz, 1088576_R2.fastq.gz)
Age: 
Telomere Length: 
Total Reads (R1+R2 combined): 113392248
3x_repeat c-strand total: 59
3x_repeat g-strand total: 38
G-strand mutations total: 6
C-strand mutations total: 14
G-strand normalizer: 44
C-strand normalizer: 73
Total mutations found: 20

Processing sample: 2256793 (files: 2256793_R1.fastq.gz, 2256793_R2.fastq.gz)
Age: 
Telomere Length: 
Total Reads (R1+R2 combined): 131979504
3x_repeat c-strand total: 46
3x_repeat g-strand total: 57
G-strand mutations total: 3
C-strand mutations total: 22
G-strand normalizer: 60
C-strand normalizer: 68
Total mutations found: 25

Processing sample: 3248160 (files: 3248160_R1.fastq.gz, 3248160_R2.fastq.gz)
Age: 
Telomere Length: 
Total Reads (R1+R2 combined): 176590488
3x_repeat c-strand total: 55
3x_repeat g-strand total: 46
G-strand mutations total: 1
C-s

## Plotting (from plotting.py)

Run all plotting pipelines. Uses `csv_path` from the generate step above. If you already have a CSV, set `csv_path` manually and run the plot cells.

In [10]:
# Derive version label from REPEAT_K (no JSON patterns file needed for plotting)
PATTERNS_VERSION = f"{REPEAT_K}x_repeat"

def _get_patterns_version(patterns_file_path=None):
    """Return the patterns version string.  Falls back to PATTERNS_VERSION (from REPEAT_K)."""
    if patterns_file_path and os.path.exists(patterns_file_path):
        try:
            with open(patterns_file_path) as f:
                return json.load(f).get('version', PATTERNS_VERSION)
        except Exception:
            pass
    return PATTERNS_VERSION

def _default_csv_path_from_patterns(patterns_file_path=None):
    """Return the default CSV output path, preferring the already-configured CSV_OUT."""
    try:
        return CSV_OUT
    except NameError:
        safe = PATTERNS_VERSION.replace(' ', '_')
        return os.path.join(_ANALYSIS_DIR, f'telomere_analysis_{safe}.csv')

# Ensure csv_path is available (set by the generate step above, or fall back)
try:
    csv_path
except NameError:
    csv_path = _default_csv_path_from_patterns()

In [11]:
def _plot_histograms_by_age_group(data, output_path):
    """Plot boxplots of mutation rate variables in 10-year age bins (2x2 grid)."""
    sns.set_style("whitegrid")
    sns.set_palette("husl")
    fig, axes = plt.subplots(2, 2, figsize=(20, 12))
    fig.suptitle('Mutation Rates by Age Groups (10-year bins)', fontsize=16, fontweight='bold')
    variables = ['total_mutations_over_total_g_strand_per_1k', 'g_strand_A>G_sum_per_1k', 'g_strand_T>G_sum_per_1k', 'g_strand_T>C_sum_per_1k']
    titles = ['Total Mutations Normalized', 'G > A Mutation Rate Normalized', 'T > G Mutation Rate Normalized', 'T > C Mutation Rate Normalized']
    for i, (var, title) in enumerate(zip(variables, titles)):
        row, col = i // 2, i % 2
        ax = axes[row, col]
        plot_data = data.dropna(subset=['Age', var])
        if len(plot_data) > 0:
            age_bins = np.arange(0, plot_data['Age'].max() + 10, 10)
            age_labels = [f'{int(b)}-{int(b+9)}' for b in age_bins[:-1]]
            plot_data = plot_data.copy()
            plot_data['Age_Group'] = pd.cut(plot_data['Age'], bins=age_bins, labels=age_labels, include_lowest=True)
            sns.boxplot(data=plot_data, x='Age_Group', y=var, ax=ax)
            ax.set_xlabel('Age Group (years)')
            ax.set_ylabel('Mutations Normalized')
            plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
        else:
            ax.text(0.5, 0.5, 'No data available', ha='center', va='center', transform=ax.transAxes, fontsize=12)
        ax.set_title(title, fontweight='bold')
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()

def _plot_mutations_per_file(data, output_path):
    """Bar plot of total mutations per file."""
    sns.set_style("whitegrid")
    mutation_columns = [col for col in data.columns if 'mutations' in col and 'per_1k' not in col]
    data_with_totals = data.copy()
    data_with_totals['Total_Mutations'] = data[mutation_columns].sum(axis=1)
    plot_data = data_with_totals.dropna(subset=['FileName', 'Total_Mutations'])
    plt.figure(figsize=(16, 10))
    if len(plot_data) > 0:
        plot_data = plot_data.sort_values('Total_Mutations', ascending=True)
        bars = plt.bar(range(len(plot_data)), plot_data['Total_Mutations'], color='steelblue', alpha=0.7, edgecolor='black', linewidth=0.5)
        plt.title('Total Number of Mutations per File', fontsize=16, fontweight='bold', pad=20)
        plt.xlabel('Files'); plt.ylabel('Total Number of Mutations')
        plt.xticks(range(len(plot_data)), plot_data['FileName'], rotation=45, ha='right')
        max_val = plot_data['Total_Mutations'].max()
        for bar in bars:
            plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max_val*0.01, f'{int(bar.get_height())}', ha='center', va='bottom', fontsize=8)
        stats_text = f"Mean: {plot_data['Total_Mutations'].mean():.0f}\nMedian: {plot_data['Total_Mutations'].median():.0f}\nMin: {plot_data['Total_Mutations'].min():.0f}\nMax: {max_val:.0f}"
        plt.text(0.02, 0.98, stats_text, transform=plt.gca().transAxes, fontsize=10, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
        plt.grid(axis='y', alpha=0.3)
    else:
        plt.text(0.5, 0.5, 'No data available', ha='center', va='center', transform=plt.gca().transAxes, fontsize=12)
        plt.title('Total Number of Mutations per File', fontweight='bold')
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    return len(plot_data)

def plot_histograms_from_csv(csv_path, output_dir=None):
    """Reproduce histogram-style plots given a telomere analysis CSV path."""
    if output_dir is None:
        output_dir = os.path.dirname(csv_path) or '.'
    os.makedirs(output_dir, exist_ok=True)
    data = pd.read_csv(csv_path)
    hist_path = os.path.join(output_dir, "histogram.png")
    per_file_path = os.path.join(output_dir, "mutations_per_file_histogram.png")
    _plot_histograms_by_age_group(data, hist_path)
    num_files = _plot_mutations_per_file(data, per_file_path)
    print(f"Histogram plot saved as '{hist_path}'")
    print(f"Mutations per file histogram saved as '{per_file_path}'")
    print(f"Processed {num_files} files")

In [12]:
def plot_trendlines(data, output_path, variables, titles, version):
    """Plot linear trendlines of mutation rate variables vs Age in 2x2 grid."""
    sns.set_style("whitegrid")
    sns.set_palette("husl")
    fig, axes = plt.subplots(2, 2, figsize=(20, 12))
    fig.suptitle(f'Mutation Rates vs Age [{version}]', fontsize=16, fontweight='bold')
    for i, (var, title) in enumerate(zip(variables, titles)):
        row, col = i // 2, i % 2
        ax = axes[row, col]
        plot_data = data.dropna(subset=['Age', var])
        if len(plot_data) > 0:
            sns.scatterplot(data=plot_data, x='Age', y=var, ax=ax, alpha=0.6)
            if len(plot_data) > 1:
                sns.regplot(data=plot_data, x='Age', y=var, ax=ax, scatter=False, line_kws={'color': 'blue', 'linestyle': '--'})
                slope, intercept, r_value, p_value, std_err = linregress(plot_data['Age'], plot_data[var])
                ax.set_title(f"{title}\nR² = {(r_value**2):.3f}", fontweight='bold')
            else:
                ax.set_title(title, fontweight='bold')
            ax.set_xlabel('Age'); ax.set_ylabel('Mutations Normalized')
        else:
            ax.text(0.5, 0.5, 'No data available', ha='center', va='center', transform=ax.transAxes, fontsize=12)
            ax.set_title(title, fontweight='bold')
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()

def plot_spearman_trendlines(data, output_path, variables, titles, version):
    """Plot Spearman correlation scatterplots vs Age in 2x2 grid."""
    sns.set_style("whitegrid")
    sns.set_palette("husl")
    fig, axes = plt.subplots(2, 2, figsize=(20, 12))
    fig.suptitle(f"Spearman's Rank Correlation: Mutation Rates vs Age [{version}]", fontsize=16, fontweight='bold')
    for i, (var, title) in enumerate(zip(variables, titles)):
        row, col = i // 2, i % 2
        ax = axes[row, col]
        plot_data = data.dropna(subset=['Age', var])
        if len(plot_data) > 1:
            corr, pval = spearmanr(plot_data['Age'], plot_data[var])
            sns.scatterplot(data=plot_data, x='Age', y=var, ax=ax, alpha=0.6)
            ax.set_title(f"{title}\nSpearman r = {corr:.2f}, p = {pval:.2g}", fontweight='bold')
        else:
            ax.text(0.5, 0.5, 'No data available', ha='center', va='center', transform=ax.transAxes, fontsize=12)
            ax.set_title(title, fontweight='bold')
        ax.set_xlabel('Age'); ax.set_ylabel('Mutations Normalized')
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()

def plot_trendlines_main(csv_path, trendline_output_path, spearman_output_path, variables, titles, patterns_file_path):
    """Main entry for 2x2 trendline and Spearman plots."""
    version = _get_patterns_version(patterns_file_path)
    data = pd.read_csv(csv_path)
    plot_trendlines(data, trendline_output_path, variables, titles, version)
    plot_spearman_trendlines(data, spearman_output_path, variables, titles, version)
    print(f"Trendline plot saved as '{trendline_output_path}'")
    print(f"Spearman correlation plot saved as '{spearman_output_path}'")

In [ ]:
def plot_mutational_signature_row(row, mutation_types, mutation_columns, output_path, version):
    sns.set_style("whitegrid")
    sns.set_palette("husl")
    bar_heights, bar_colors, bar_labels = [], [], []
    all_columns = []
    for mut_type, contexts in mutation_columns.items():
        for context, cols in contexts.items():
            existing_cols = [col for col in cols if col in row.index]
            all_columns.extend(existing_cols)
    if not all_columns:
        print(f"Warning: No valid mutation columns found for {row.get('FileName', 'unknown')}")
        return
    total_mutations = row[all_columns].sum()
    for mut_label, color in mutation_types:
        contexts = mutation_columns[mut_label]
        for context_name, cols in contexts.items():
            for i, col in enumerate(cols):
                if col in row.index:
                    value = row[col]
                    percentage = (value / total_mutations) * 100 if total_mutations > 0 else 0
                    bar_heights.append(percentage)
                    bar_colors.append(color)
                    bar_labels.append(f"{mut_label} {context_name} pos{i+1}")
    if not bar_heights:
        print(f"Warning: No valid data found for {row.get('FileName', 'unknown')}")
        return
    x = np.arange(len(bar_heights))
    fig, ax = plt.subplots(figsize=(16, 10))
    sns.barplot(x=x, y=bar_heights, palette=bar_colors, ax=ax, edgecolor='black', linewidth=0.5)
    for i, label in enumerate(bar_labels):
        ax.text(i, -max(bar_heights)*0.02, label, ha='center', va='center', color='black', fontsize=9, fontweight='normal', rotation=45)
    ax.set_xticks([])
    ax.set_yticks(np.linspace(0, max(bar_heights), 6))
    ax.set_xlim(-0.5, len(bar_heights) - 0.5)
    ax.set_ylim(-max(bar_heights)*0.1, max(bar_heights) + max(bar_heights)*0.2)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False); ax.spines['left'].set_visible(False)
    ax.yaxis.grid(True, linestyle='--', alpha=0.3)
    ax.set_axisbelow(True)
    ax.set_ylabel('Percentage of Single Base Modifications', fontsize=14, fontweight='bold')
    age = row['Age'] if 'Age' in row else 'N/A'
    filename = row['FileName'] if 'FileName' in row else 'sample'
    ax.set_title(f"Mutational Signatures by Position and Strand Context [{version}]\nFile: {filename} | Age: {age} years", fontsize=18, fontweight='bold', pad=30)
    plt.tight_layout(rect=[0, 0.15, 1, 0.95])
    plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()

def plot_mutational_signatures(csv_path, patterns_file_path):
    sns.set_theme(style="whitegrid", font_scale=1.1)
    version = _get_patterns_version(patterns_file_path)
    df = pd.read_csv(csv_path)
    mutation_types = [('C>A','blue'),('C>G','black'),('C>T','red'),('G>A','gray'),('G>C','green'),('G>T','pink')]
    mutation_columns = {
        'C>A': {'C-strand': ['c_strand_mutations_C>A_c1','c_strand_mutations_C>A_c2','c_strand_mutations_C>A_c3'], 'G-strand': ['g_strand_mutations_G>T_g1','g_strand_mutations_G>T_g2','g_strand_mutations_G>T_g3']},
        'C>G': {'C-strand': ['c_strand_mutations_C>G_c1','c_strand_mutations_C>G_c2','c_strand_mutations_C>G_c3'], 'G-strand': ['g_strand_mutations_G>C_g1','g_strand_mutations_G>C_g2','g_strand_mutations_G>C_g3']},
        'C>T': {'C-strand': ['c_strand_mutations_C>T_c1','c_strand_mutations_C>T_c2','c_strand_mutations_C>T_c3'], 'G-strand': ['g_strand_mutations_G>A_g1','g_strand_mutations_G>A_g2','g_strand_mutations_G>A_g3']},
        'G>A': {'G-strand': ['g_strand_mutations_G>A_g1','g_strand_mutations_G>A_g2','g_strand_mutations_G>A_g3'], 'C-strand': ['c_strand_mutations_C>G_c1','c_strand_mutations_C>G_c2','c_strand_mutations_C>G_c3']},
        'G>C': {'G-strand': ['g_strand_mutations_G>C_g1','g_strand_mutations_G>C_g2','g_strand_mutations_G>C_g3'], 'C-strand': ['c_strand_mutations_C>T_c1','c_strand_mutations_C>T_c2','c_strand_mutations_C>T_c3']},
        'G>T': {'G-strand': ['g_strand_mutations_G>T_g1','g_strand_mutations_G>T_g2','g_strand_mutations_G>T_g3'], 'C-strand': ['c_strand_mutations_C>A_c1','c_strand_mutations_C>A_c2','c_strand_mutations_C>A_c3']},
    }
    os.makedirs('plots', exist_ok=True)
    for idx, row in df.iterrows():
        filename = str(row['FileName']) if 'FileName' in row else f'sample_{idx}'
        filename_base = os.path.splitext(filename)[0]
        output_path = os.path.join('plots', f'{filename_base}.png')
        plot_mutational_signature_row(row, mutation_types, mutation_columns, output_path, version)

def plot_mutational_signatures_main(patterns_file_path):
    csv_path = _default_csv_path_from_patterns(patterns_file_path)
    plot_mutational_signatures(csv_path, patterns_file_path)
    print("Mutational signature plots saved in 'plots/' directory")

In [13]:
def plot_spearman_with_age(csv_path, patterns_file_path):
    """Plot scatter of each numeric column vs Age with Spearman correlation."""
    sns.set_theme(style="whitegrid", font_scale=1.1)
    df = pd.read_csv(csv_path)
    output_dir = "spearman's plots"
    os.makedirs(output_dir, exist_ok=True)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if 'Age' not in numeric_cols:
        print("No 'Age' column found.")
        return
    numeric_cols = [c for c in numeric_cols if c != 'Age']
    df = df.dropna(subset=['Age'])
    spearman_results = []
    version = _get_patterns_version(patterns_file_path)
    for col in numeric_cols:
        sub_df = df.dropna(subset=[col])
        if sub_df.shape[0] < 2:
            continue
        x, y = sub_df['Age'], sub_df[col]
        corr, pval = stats.spearmanr(x, y)
        spearman_results.append({'Column': col, 'Spearman_r': corr, 'p_value': pval})
        plt.figure(figsize=(8, 6))
        ax = sns.scatterplot(x=x, y=y)
        if len(x) > 1:
            sns.regplot(x=x, y=y, scatter=False, ci=None, line_kws={'color': 'red', 'linestyle': '--'}, ax=ax)
        ax.set_xlabel('Age (years)'); ax.set_ylabel(col)
        ax.set_title(f"Spearman's ρ = {corr:.2f} (p={pval:.2g})\n{col} vs Age [{version}]", fontsize=14)
        plt.tight_layout()
        safe_col = col.replace('/', '_').replace(' ', '_').replace('>', 'to').replace('<', 'lt').replace(':', '_')
        plt.savefig(os.path.join(output_dir, f"{safe_col}_vs_Age.png"), dpi=200)
        plt.close()
    pd.DataFrame(spearman_results).to_csv(os.path.join(output_dir, "spearman_results.csv"), index=False)

def plot_spearman_with_age_main(patterns_file_path):
    csv_path = _default_csv_path_from_patterns(patterns_file_path)
    plot_spearman_with_age(csv_path, patterns_file_path)
    print("Spearman plots saved in \"spearman's plots/\" directory")

def plot_mutation_r_heatmap(csv_path, target_col='Age', patterns_file_path=None):
    """Heatmap of Spearman r values between per_1k mutation columns and target."""
    df = pd.read_csv(csv_path)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if target_col not in numeric_cols:
        print(f"No '{target_col}' column found.")
        return
    mutation_cols = [c for c in numeric_cols if 'per_1k' in c or 'per1k' in c]
    if 'total_mutations_per_1k_reads' in df.columns and 'total_mutations_per_1k_reads' not in mutation_cols:
        mutation_cols.append('total_mutations_per_1k_reads')
    total_mut_col = next((c for c in numeric_cols if 'total_mutation' in c and c not in mutation_cols), None)
    if total_mut_col:
        mutation_cols.append(total_mut_col)
    frameshift_cols = [c for c in numeric_cols if 'frameshift' in c.lower() and 'per_1k' in c]
    mutation_cols.extend([c for c in frameshift_cols if c not in mutation_cols])
    if not mutation_cols:
        print("No per_1k mutation columns found.")
        return
    df = df.dropna(subset=[target_col])
    r_values = []
    for col in mutation_cols:
        sub_df = df.dropna(subset=[col])
        r_values.append(stats.spearmanr(sub_df[col], sub_df[target_col])[0] if sub_df.shape[0] >= 2 else float('nan'))
    r_df = pd.DataFrame({'Mutation': mutation_cols, 'Spearman_r': r_values}).set_index('Mutation')
    plt.figure(figsize=(max(8, len(mutation_cols)*0.4), 2.5))
    sns.heatmap(r_df.T, annot=True, cmap='coolwarm', center=0, cbar_kws={'label': "Spearman's r"})
    version = _get_patterns_version(patterns_file_path) if patterns_file_path else 'unknown'
    plt.title(f"Spearman r values: Normalized Mutations vs {target_col} [{version}]")
    plt.yticks(rotation=0)
    plt.tight_layout()
    output_dir = "spearman's plots"
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, f"mutation_r_heatmap_vs_{target_col}.png")
    plt.savefig(output_path, dpi=200)
    plt.close()
    print(f"Mutation r heatmap saved as {output_path}")

def plot_pairwise_r_heatmap(csv_path, patterns_file_path=None):
    """Pairwise Spearman r heatmap between per_1k, Age, telomere columns."""
    df = pd.read_csv(csv_path)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    mutation_cols = [c for c in numeric_cols if 'per_1k' in c or 'per1k' in c]
    if 'total_mutations_per_1k_reads' in df.columns and 'total_mutations_per_1k_reads' not in mutation_cols:
        mutation_cols.append('total_mutations_per_1k_reads')
    total_mut_col = next((c for c in numeric_cols if 'total_mutation' in c and c not in mutation_cols), None)
    if total_mut_col:
        mutation_cols.append(total_mut_col)
    frameshift_cols = [c for c in numeric_cols if 'frameshift' in c.lower() and 'per_1k' in c]
    mutation_cols.extend([c for c in frameshift_cols if c not in mutation_cols])
    if 'Total_Reads' in df.columns and 'Total_Reads' in numeric_cols and 'Total_Reads' not in mutation_cols:
        mutation_cols.append('Total_Reads')
    if 'Age' in df.columns and 'Age' in numeric_cols and 'Age' not in mutation_cols:
        mutation_cols.append('Age')
    telomere_cols = [c for c in df.columns if 'telomere' in c.lower() and c in numeric_cols and c not in mutation_cols]
    mutation_cols.extend(telomere_cols)
    if not mutation_cols:
        print("No relevant columns for pairwise heatmap.")
        return
    n = len(mutation_cols)
    r_matrix = np.zeros((n, n))
    for i, col1 in enumerate(mutation_cols):
        for j, col2 in enumerate(mutation_cols):
            sub_df = df[[col1, col2]].dropna()
            r_val = stats.spearmanr(sub_df[col1], sub_df[col2])[0] if sub_df.shape[0] >= 2 else np.nan
            r_matrix[i, j] = float(r_val) if isinstance(r_val, numbers.Number) and not isinstance(r_val, (list, tuple, np.ndarray)) else np.nan
    mask = np.eye(n, dtype=bool)
    fig_width, fig_height = max(10, n*0.7), max(8, n*0.7)
    plt.figure(figsize=(fig_width, fig_height))
    ax = sns.heatmap(r_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, mask=mask,
                     xticklabels=mutation_cols, yticklabels=mutation_cols, cbar_kws={'label': "Spearman's r"}, annot_kws={"size": 8})
    for i in range(n):
        ax.add_patch(plt.Rectangle((i, i), 1, 1, fill=False, edgecolor='black', lw=2, hatch='xx'))
    version = _get_patterns_version(patterns_file_path) if patterns_file_path else 'unknown'
    plt.title(f"Pairwise Spearman r Heatmap (Normalized Mutations, Age, Telomere) [{version}]", fontsize=14)
    plt.xticks(rotation=45, ha='right', fontsize=9)
    plt.yticks(fontsize=9)
    plt.tight_layout()
    output_dir = "spearman's plots"
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, "pairwise_mutation_r_heatmap.png")
    plt.savefig(output_path, dpi=200)
    plt.close()
    print(f"Pairwise mutation r heatmap saved as {output_path}")

def plot_pairwise_r_heatmap_main(patterns_file_path):
    csv_path = _default_csv_path_from_patterns(patterns_file_path)
    plot_pairwise_r_heatmap(csv_path, patterns_file_path=patterns_file_path)

## Run All Plots (equivalent to: main.py run / main.py plot)

Run the cell below to execute the full plotting pipeline: histograms, mutational signatures, Spearman correlations, pairwise heatmap, trendlines, and curve fitting.

In [14]:
# Run full plotting pipeline
print("[2/2] Running plots ...")
os.makedirs(_HISTOGRAM_DIR, exist_ok=True)
plot_histograms_from_csv(csv_path, output_dir=_HISTOGRAM_DIR)

# Spearman correlations (version label comes from PATTERNS_VERSION derived from REPEAT_K)
plot_spearman_with_age(csv_path, patterns_file_path=None)
print("Spearman plots saved in \"spearman's plots/\" directory")

plot_pairwise_r_heatmap(csv_path, patterns_file_path=None)

# Trendlines
def _unique_output_path(directory, base_name, ext):
    os.makedirs(directory, exist_ok=True)
    timestamp = datetime.now().strftime('%Y-%m-%d_%H%M%S')
    return os.path.join(directory, f"{base_name}_{timestamp}.{ext}")

_TRENDLINE_VARIABLES = [
    "total_mutations_over_total_g_strand_per_1k",
    "g_strand_T>C_sum_per_1k",
    "g_strand_G>T_sum_per_1k",
    "g_strand_T>G_sum_per_1k",
]
_TRENDLINE_TITLES = [
    "Total Mutations Normalized",
    "T > C Mutation Rate Normalized",
    "G > T Mutation Rate Normalized",
    "T > G Mutation Rate Normalized",
]
trendline_output = _unique_output_path(_TRENDLINES_DIR, "trendline", "png")
spearman_output = _unique_output_path(_SPEARMAN_CORRELATIONS_DIR, "spearman_correlation", "png")
plot_trendlines_main(csv_path, trendline_output, spearman_output, _TRENDLINE_VARIABLES, _TRENDLINE_TITLES, None)

print("\nAll plotting complete.")

[2/2] Running plots ...

Histogram plot saved as '/Users/akhilpeddikuppa/FieldLab/GreiderCodeSearch/analysis/histograms/histogram.png'

Mutations per file histogram saved as '/Users/akhilpeddikuppa/FieldLab/GreiderCodeSearch/analysis/histograms/mutations_per_file_histogram.png'

Processed 153 files

Spearman plots saved in "spearman's plots/" directory

Pairwise mutation r heatmap saved as spearman's plots/pairwise_mutation_r_heatmap.png

Trendline plot saved as '/Users/akhilpeddikuppa/FieldLab/GreiderCodeSearch/analysis/trendlines/trendline_2026-02-18_140130.png'

Spearman correlation plot saved as '/Users/akhilpeddikuppa/FieldLab/GreiderCodeSearch/analysis/spearman_correlations/spearman_correlation_2026-02-18_140130.png'



All plotting complete.
